# High-dimensional gene-expression ML

A short walk through the research note: PCA and clustering, cross-validated classification of three models, and FDR-controlled differential expression on the committed Golub leukemia sample (72 samples, top 1000 most variable genes, classes ALL and AML). This runs fully offline. Point the loader at `data/leukemia.csv` from `scripts/download_data.py` to reproduce the whole-genome numbers.

In [1]:
from hdgenomics import (
    load_leukemia, standardize, top_variance_genes, pca_embedding,
    kmeans_clusters, compare_models, differential_expression,
)
from hdgenomics.dimreduce import cluster_agreement

data = load_leukemia('../data/leukemia_sample.csv')
keep = top_variance_genes(data.X, 1000)
xs, _ = standardize(data.X[:, keep])
data.n_samples, data.n_genes, data.class_names

(72, 1000, ['ALL', 'AML'])

## Does unsupervised structure recover the subtypes?

In [2]:
emb, evr = pca_embedding(xs, 2)
clusters = kmeans_clusters(emb, 2)
print('PCA explained variance (PC1, PC2):', evr.round(3))
print('K-means vs true labels ARI:', round(cluster_agreement(data.y, clusters), 3))

PCA explained variance (PC1, PC2): [0.165 0.12 ]
K-means vs true labels ARI: 0.016


The adjusted Rand index is near zero: the dominant axes of variation are not the ALL/AML contrast, so k-means over the most variable genes does not recover the diagnostic split.

## Can supervised models separate the subtypes?

In [3]:
for name, m in compare_models(xs, data.y).items():
    print(f"{name:>7}: acc {m['accuracy_mean']:.3f} +/- {m['accuracy_std']:.3f}, macro F1 {m['f1_macro_mean']:.3f}")

 logreg: acc 0.958 +/- 0.057, macro F1 0.955
    svm: acc 0.944 +/- 0.053, macro F1 0.940
     rf: acc 0.986 +/- 0.029, macro F1 0.984


## Which genes are differentially expressed?

In [4]:
de = differential_expression(data.X, data.y, data.gene_names)
print('Genes significant at FDR 0.05:', int(de.significant.sum()), 'of', data.n_genes)

Genes significant at FDR 0.05: 362 of 1000
